# Conventions and identities

Companion notebook to *Advanced Calculus for Machine Learning*, across
chapters rather than for one of them.

The other notebooks check the numbers printed in the chapters. This one
checks the **conventions and identities** that the proofs rest on: the action
of a permutation on a multilinear form, the normalization of the wedge
product, the change of basis of the components of a tensor, the Hodge star,
the signs in the formula for the exterior derivative and in Cartan's formula,
and a few identities used in passing. Each is re-implemented from its
statement and compared with the book, with an assertion.

A convention is exactly the kind of thing that is easy to get backwards and
hard to catch by rereading, because both versions look reasonable on the
page. Three of the cells below exist only because of that risk.

In [1]:
import itertools
import math

import sympy as sp
import torch

torch.set_default_dtype(torch.float64)
torch.manual_seed(0)
print(torch.__version__, sp.__version__)

2.15.0+cu132 1.14.0


## 1. Permutations acting on $k$-linear forms (§21.2)

The book writes $(\sigma T)(v_1,\dots,v_k) = T(v_{\sigma(1)},\dots,v_{\sigma(k)})$
and states that $\tau(\sigma T) = (\tau\sigma)T$.

`torch.permute` (like `numpy.transpose`) applies the *inverse* permutation to
the axes, so the implementation below uses `argsort`. The first cell checks
that implementation against the definition on explicit vectors, before it is
used to check anything else.

In [2]:
def act(sigma, T):
    "sigma T, with (sigma T)(v_1..v_k) = T(v_{sigma(1)},..,v_{sigma(k)})."
    inv = torch.argsort(torch.tensor(sigma))
    return T.permute(tuple(inv.tolist()))


n, k = 4, 3
T = torch.randn(n, n, n)
vs = [torch.randn(n) for _ in range(k)]


def value(F, vectors):
    return torch.einsum("ijk,i,j,k->", F, *vectors)


for s in itertools.permutations(range(k)):
    want = value(T, [vs[s[r]] for r in range(k)])
    assert torch.isclose(value(act(s, T), vs), want), s
print("the implementation of sigma T agrees with the definition")

the implementation of sigma T agrees with the definition


In [3]:
# tau(sigma T) = (tau sigma) T, and not the other way round
for s, t in itertools.product(itertools.permutations(range(k)), repeat=2):
    composite = tuple(t[s[i]] for i in range(k))      # (tau sigma)(i) = tau(sigma(i))
    assert torch.allclose(act(t, act(s, T)), act(composite, T))
    if s != t:
        other = tuple(s[t[i]] for i in range(k))      # (sigma tau), the wrong one
        if other != composite:
            assert not torch.allclose(act(t, act(s, T)), act(other, T))
print("tau(sigma T) = (tau sigma) T")

tau(sigma T) = (tau sigma) T


## 2. The sign of a permutation (Lemma 21.6)

$\operatorname{sgn}(\sigma) = (-1)^N$ with $N$ the number of pairs $i < j$
such that $\sigma(i) > \sigma(j)$. The lemma says this is multiplicative and
that an exchange has sign $-1$.

In [4]:
def sgn(p):
    n = len(p)
    inv = sum(1 for i in range(n) for j in range(i + 1, n) if p[i] > p[j])
    return (-1) ** inv


for m in (3, 4, 5):
    perms = list(itertools.permutations(range(m)))
    for s, t in itertools.product(perms, repeat=2):
        assert sgn(tuple(s[t[i]] for i in range(m))) == sgn(s) * sgn(t)
    for i in range(m):
        for j in range(i + 1, m):
            e = list(range(m))
            e[i], e[j] = e[j], e[i]
            assert sgn(tuple(e)) == -1
print("sgn is multiplicative and an exchange has sign -1")

sgn is multiplicative and an exchange has sign -1


## 3. Antisymmetrization and the wedge product (§21.2, §21.3)

$\mathcal A(T) = \sum_\sigma \operatorname{sgn}(\sigma)\,\sigma T$ is
alternating, $\mathcal A(\omega) = k!\,\omega$ when $\omega$ already is, and
$\omega\wedge\eta = \frac1{k!\,l!}\mathcal A(\omega\otimes\eta)$.

The four rules of Theorem 21.11 are checked, and so is the step
$\mathcal A(\mathcal A(T)\otimes S) = m!\,\mathcal A(T\otimes S)$ on which its
proof of associativity turns.

In [5]:
def A(T):
    k = T.dim()
    out = torch.zeros_like(T)
    for s in itertools.permutations(range(k)):
        out = out + sgn(s) * act(s, T)
    return out


def tensor(a, b):
    return torch.tensordot(a, b, dims=0)


def wedge(w, e):
    k, l = w.dim(), e.dim()
    return A(tensor(w, e)) / (math.factorial(k) * math.factorial(l))


def alternating(n, k):
    return A(torch.randn(*([n] * k))) / math.factorial(k)


n = 4
for k in (2, 3):
    T = torch.randn(*([n] * k))
    for t in itertools.permutations(range(k)):
        assert torch.allclose(act(t, A(T)), sgn(t) * A(T))
        assert torch.allclose(A(act(t, T)), sgn(t) * A(T))
    w = alternating(n, k)
    assert torch.allclose(A(w), math.factorial(k) * w)
print("A(T) is alternating and A(omega) = k! omega")

A(T) is alternating and A(omega) = k! omega


In [6]:
phi, psi = torch.randn(n), torch.randn(n)
u, v = torch.randn(n), torch.randn(n)
assert torch.isclose(
    torch.einsum("ij,i,j->", wedge(phi, psi), u, v),
    (phi @ u) * (psi @ v) - (phi @ v) * (psi @ u),
)

a1, a2, a3 = alternating(n, 1), alternating(n, 1), alternating(n, 2)
assert torch.allclose(wedge(wedge(a1, a2), a3), wedge(a1, wedge(a2, a3)))
assert torch.allclose(
    wedge(wedge(a1, a2), a3),
    A(tensor(tensor(a1, a2), a3)) / (1 * 1 * math.factorial(2)),
)
for kk, ll in [(1, 2), (2, 1), (2, 2), (1, 3), (3, 1)]:
    x, y = alternating(n, kk), alternating(n, ll)
    assert torch.allclose(wedge(y, x), (-1) ** (kk * ll) * wedge(x, y))
print("wedge: the formula on covectors, associativity, and the sign (-1)^{kl}")

wedge: the formula on covectors, associativity, and the sign (-1)^{kl}


In [7]:
for m, l in [(1, 2), (2, 1), (2, 2)]:
    Tm, Sl = torch.randn(*([n] * m)), torch.randn(*([n] * l))
    assert torch.allclose(
        A(tensor(A(Tm), Sl)), math.factorial(m) * A(tensor(Tm, Sl))
    )
print("A(A(T) (x) S) = m! A(T (x) S)")

A(A(T) (x) S) = m! A(T (x) S)


## 4. Components under a change of basis (Theorem 21.4)

With $e' = eP$, the components of a tensor of type $(p, q)$ pick up one
$P^{-1}$ per slot for covectors and one $P$ per slot for vectors. Getting
these two the wrong way round is the classical slip, so the formula is
compared with a direct evaluation of the tensor on the new basis.

In [8]:
m = 3
Tc = torch.randn(m, m, m)                       # type (1, 2)
P = torch.randn(m, m)
while abs(torch.linalg.det(P)) < 1e-2:
    P = torch.randn(m, m)
Pi = torch.linalg.inv(P)

by_formula = torch.einsum("ia,bj,ck,abc->ijk", Pi, P, P, Tc)
direct = torch.einsum("abc,ia,bj,ck->ijk", Tc, Pi, P, P)   # e'* = P^-1 e*, e' = eP
assert torch.allclose(by_formula, direct)
print("P^-1 on the covector slots, P on the vector slots")

P^-1 on the covector slots, P on the vector slots


## 5. Minors and Cauchy–Binet (Proposition 21.18)

In [9]:
def minor(M, rows, cols):
    return torch.linalg.det(M[list(rows)][:, list(cols)])


Amat, Bmat = torch.randn(4, 5), torch.randn(5, 3)
kk = 2
for J in itertools.combinations(range(4), kk):
    for I in itertools.combinations(range(3), kk):
        rhs = sum(
            minor(Amat, J, K) * minor(Bmat, K, I)
            for K in itertools.combinations(range(5), kk)
        )
        assert torch.isclose(minor(Amat @ Bmat, J, I), rhs)

M = torch.randn(5, 3)
assert torch.isclose(
    torch.linalg.det(M.T @ M),
    sum(minor(M, K, (0, 1, 2)) ** 2 for K in itertools.combinations(range(5), 3)),
)
print("Cauchy-Binet, and det(M^T M) as a sum of squared minors")

Cauchy-Binet, and det(M^T M) as a sum of squared minors


## 6. The Hodge star (Theorem 21.21)

$\star$ is defined by $\alpha\wedge\star\beta = \langle\alpha,\beta\rangle\,
\mathrm{vol}$. On a positive orthonormal basis it sends $o^*_I$ to
$\pm o^*_{I^c}$, and $\star\star = (-1)^{k(n-k)}$.

In [10]:
def star_matrix(n, k):
    idx = list(itertools.combinations(range(n), k))
    idxc = list(itertools.combinations(range(n), n - k))
    S = torch.zeros(len(idxc), len(idx))
    for j, I in enumerate(idx):
        Ic = tuple(sorted(set(range(n)) - set(I)))
        S[idxc.index(Ic), j] = sgn(tuple(list(I) + list(Ic)))
    return S, idx, idxc


for nn in (3, 4, 5):
    for k in range(nn + 1):
        S1, idx, _ = star_matrix(nn, k)
        S2, _, _ = star_matrix(nn, nn - k)
        assert torch.allclose(S2 @ S1, (-1) ** (k * (nn - k)) * torch.eye(len(idx)))
print("star star = (-1)^{k(n-k)}")

star star = (-1)^{k(n-k)}


In [11]:
# alpha ^ star beta = <alpha, beta> vol, for 1-forms of R^3
nn = 3
basis = [torch.eye(nn)[i] for i in range(nn)]
vol = A(tensor(tensor(basis[0], basis[1]), basis[2]))
S1, idx, idxc = star_matrix(nn, 1)
for i in range(nn):
    star_i = torch.zeros(nn, nn)
    for r, Ic in enumerate(idxc):
        if S1[r, i] != 0:
            star_i = star_i + S1[r, i] * A(tensor(basis[Ic[0]], basis[Ic[1]]))
    for j in range(nn):
        expected = (1.0 if i == j else 0.0) * vol
        assert torch.allclose(wedge(basis[j], star_i), expected)
print("alpha ^ star beta = <alpha, beta> vol")

alpha ^ star beta = <alpha, beta> vol


## 7. The exterior derivative and Cartan's formula (Theorems 22.12 and 25.15)

These two carry alternating signs, and the vectors in Theorem 22.12 are
numbered from $0$. They are checked symbolically on $\mathbb R^3$, with
polynomial coefficients, against the exterior derivative computed from the
components.

In [12]:
x = sp.symbols("x0:3")
nn = 3


def d_form(comp):
    "Exterior derivative, as a dict on increasing multi-indices."
    out = {}
    for I, f in comp.items():
        for i in range(nn):
            if i in I:
                continue
            J = tuple(sorted(I + (i,)))
            perm = [i] + list(I)
            order = sorted(range(len(perm)), key=lambda t: perm[t])
            out[J] = out.get(J, 0) + sgn(tuple(order)) * sp.diff(f, x[i])
    return out


def eval_form(comp, vectors):
    total = 0
    for I, f in comp.items():
        M = sp.Matrix([[vec[c] for c in I] for vec in vectors])
        total += f * M.det()
    return sp.expand(total)


def interior(comp, field):
    "i_X omega, by Definition 22.x: (i_X omega)(v_2..v_k) = omega(X, v_2..v_k)."
    out = {}
    for I, f in comp.items():
        for pos, i in enumerate(I):
            J = tuple(c for c in I if c != i)
            out[J] = out.get(J, 0) + (-1) ** pos * f * field[i]
    return out


def random_form(k, degree=2):
    comp = {}
    for I in itertools.combinations(range(nn), k):
        comp[I] = sum(
            sp.Integer(int(torch.randint(-3, 4, (1,)))) * x[i] ** d
            for i in range(nn)
            for d in range(1, degree + 1)
        ) + sp.Integer(int(torch.randint(-3, 4, (1,))))
    return comp

In [13]:
# Theorem 22.12: d omega (v_0,..,v_k) = sum_j (-1)^j (D omega (p) v_j)(.. hat v_j ..)
for k in (1, 2):
    comp = random_form(k)
    vs = [[sp.Integer(int(c)) for c in torch.randint(-3, 4, (nn,))] for _ in range(k + 1)]
    left = eval_form(d_form(comp), vs)
    right = 0
    for j in range(k + 1):
        rest = [vs[r] for r in range(k + 1) if r != j]
        dcomp_j = {
            I: sum(sp.diff(f, x[i]) * vs[j][i] for i in range(nn))
            for I, f in comp.items()
        }
        right += (-1) ** j * eval_form(dcomp_j, rest)
    assert sp.simplify(left - right) == 0, k
print("Theorem 22.12, for a 1-form and a 2-form of R^3")

Theorem 22.12, for a 1-form and a 2-form of R^3


In [14]:
# Theorem 25.15: L_X omega = d(i_X omega) + i_X d omega
for k in (1, 2, 3):
    comp = random_form(k, degree=1)
    X = [
        sum(sp.Integer(int(torch.randint(-2, 3, (1,)))) * x[i] for i in range(nn))
        + sp.Rational(1, 2)
        for _ in range(nn)
    ]
    DX = sp.Matrix(nn, nn, lambda i, j: sp.diff(X[i], x[j]))
    vs = [[sp.Integer(int(c)) for c in torch.randint(-2, 3, (nn,))] for _ in range(k)]

    # (1) of the theorem
    lie = eval_form(
        {I: sum(sp.diff(f, x[i]) * X[i] for i in range(nn)) for I, f in comp.items()}, vs
    )
    for j in range(k):
        moved = [list(vs[r]) for r in range(k)]
        moved[j] = list(DX * sp.Matrix(vs[j]))
        lie += eval_form(comp, moved)

    # (2) of the theorem
    cartan = eval_form(d_form(interior(comp, X)), vs) if k >= 1 else 0
    if k + 1 <= nn:
        cartan += eval_form(interior(d_form(comp), X), vs)
    assert sp.simplify(sp.expand(lie) - sp.expand(cartan)) == 0, k
print("Cartan's formula, for k = 1, 2, 3 on R^3")

Cartan's formula, for k = 1, 2, 3 on R^3


## 8. Four identities used in passing

Lemma A.5 counts the integers $m$ with $a\leq m/N < b$ as
$\lceil Nb\rceil - \lceil Na\rceil$; the pitfall of §6.3 needs a function whose
directional derivatives all vanish at a point where it is not continuous; the
scores thread of Chapter 22 reads $\mathrm d\alpha(u, v) = v^\top Ju - u^\top Jv$
and its mean square as $\lVert J - J^\top\rVert_F^2$; and Theorems 3.15 and
3.16 give the adjoint as $M^{-1}A^\top N$.

In [15]:
# Lemma A.5
for N in range(1, 12):
    for _ in range(200):
        a, b = sorted(float(z) for z in torch.rand(2) * 6 - 3)
        count = sum(1 for mm in range(-60, 61) if a <= mm / N < b)
        assert count == math.ceil(N * b) - math.ceil(N * a)
print("Lemma A.5: ceil(Nb) - ceil(Na)")

Lemma A.5: ceil(Nb) - ceil(Na)


In [16]:
# Pitfall of 6.3
xx, yy, t = sp.symbols("x y t", real=True)
f = xx**3 * yy / (xx**6 + yy**2)
for hx, hy in [(1, 0), (0, 1), (1, 1), (2, -3), (-1, 5)]:
    assert sp.limit(sp.simplify(f.subs({xx: t * hx, yy: t * hy}) / t), t, 0) == 0
assert sp.simplify(f.subs({yy: xx**3})) == sp.Rational(1, 2)
print("all directional derivatives vanish, and f = 1/2 along y = x^3")

all directional derivatives vanish, and f = 1/2 along y = x^3


In [17]:
# Scores thread of Chapter 22
d = 5
J = torch.randn(d, d)
u, v = torch.randn(d), torch.randn(d)
assert torch.isclose(v @ J @ u - u @ J @ v, v @ (J - J.T) @ u)

samples = torch.randn(200_000, 2, d)
vals = torch.einsum("si,ij,sj->s", samples[:, 1], J, samples[:, 0]) - torch.einsum(
    "si,ij,sj->s", samples[:, 0], J, samples[:, 1]
)
empirical, exact = (vals**2).mean(), ((J - J.T) ** 2).sum()
assert abs(empirical - exact) / exact < 0.03
print(f"mean of d alpha(u, v)^2: {empirical:.3f} sampled, {exact:.3f} exact")

mean of d alpha(u, v)^2: 34.164 sampled, 34.428 exact


In [18]:
# Theorems 3.15 and 3.16
dv, dw = 4, 3
Mg = torch.randn(dv, dv); Mg = Mg @ Mg.T + dv * torch.eye(dv)
Ng = torch.randn(dw, dw); Ng = Ng @ Ng.T + dw * torch.eye(dw)
Aop = torch.randn(dw, dv)
adjoint = torch.linalg.inv(Mg) @ Aop.T @ Ng
for _ in range(200):
    xv, yw = torch.randn(dv), torch.randn(dw)
    assert torch.isclose((Aop @ xv) @ Ng @ yw, xv @ Mg @ (adjoint @ yw))
print("the adjoint is M^-1 A^T N")

the adjoint is M^-1 A^T N


All the assertions above pass, so every convention and identity collected here
is the one printed in the book.